# 05d – Random forest

| | |
|---|---|
| **Auteur(s)** | Kobe Vandenberk |
| **Wat** | Een random forest trainen en tunen met een randomized search, met dezelfde week-gebaseerde cross-validatie. |
| **Input** | `data/model_table.parquet` |
| **Output** | `models/metrics/random_forest.json`, `models/predictions/random_forest.parquet` |
| **GenAI** | Code en tekst opgesteld met hulp van Claude; keuzes en resultaten nagekeken door Kobe. |

## Waarom dit model?
- Een random forest middelt **veel diepe beslissingsbomen**, elk getraind op een andere bootstrap-steekproef en met willekeurige subsets van kenmerken. Het middelen verlaagt de variantie van één diepe boom (die zou overfitten).
- Bomen vinden **interacties zelf** (zone × uur × weekend × regen) zonder dat we die moeten bouwen, in tegenstelling tot de GLM in 05c.
- Het is het "andere" ensemble-type naast boosting (05e): **bagging** (parallel, onafhankelijke bomen) vs. **boosting** (sequentieel, elke boom corrigeert de vorige). Zo vergelijken we de twee families.

**Zone als getal:** scikit-learn's random forest kent geen categorische kenmerken; zone staat er als getal 0–29. De zones zijn genummerd van druk naar rustig (zie 04), dus een split "zone ≤ 5" betekent "de drukste zones" en is zinvol. Bomen kunnen bovendien meerdere splits combineren om losse zones af te zonderen.

**Verliesfunctie:** we gebruiken de standaard `squared_error`. De `poisson`-criterium bestaat ook, maar is op 200.000 rijen veel trager; een boom voorspelt toch gemiddelden van tellingen (nooit negatief).

## 0. Imports en data

In [1]:
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

import citibike as cb

table = cb.load_model_table()
X_train, y_train, g_train, X_test, y_test = cb.train_test(table)

## 1. Standaardinstellingen
Eerst zonder tuning (100 bomen, onbeperkte diepte), als vertrekpunt.

In [2]:
default = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=cb.SEED)
cv_default = cb.cv_scores(default, X_train, y_train, g_train)
cv_default

{'cv_mae': [27.9717, 30.8699, 24.6562, 30.1848, 27.0317],
 'cv_mae_mean': 28.1429,
 'cv_poisson_deviance': [10.1041, 14.2149, 8.8326, 12.6475, 9.4695],
 'cv_fit_seconds': 22.0}

## 2. Randomized search
Wat we tunen en waarom:
| Hyperparameter | Effect |
|---|---|
| `min_samples_leaf` | minimum aantal uren per blad: groter = gladder, minder ruis volgen |
| `max_features` | aandeel kenmerken per split: kleiner = meer verschillende bomen (minder correlatie) |
| `max_depth` | maximale diepte: begrenst de complexiteit |
| `max_samples` | grootte van de bootstrap-steekproef per boom: kleiner = sneller en diverser |

### Ronde 1: breed zoeken
12 willekeurige combinaties × 3 folds (op week gegroepeerd), met 150 bomen. We verwachtten dat grotere bladeren en minder kenmerken per split overfitting zouden tegengaan.

In [3]:
search = RandomizedSearchCV(
    RandomForestRegressor(n_estimators=150, n_jobs=-1, random_state=cb.SEED),
    {"min_samples_leaf": randint(1, 30), "max_features": uniform(0.3, 0.7),
     "max_depth": [None, 12, 18, 25], "max_samples": uniform(0.3, 0.7)},
    n_iter=12, cv=GroupKFold(3), scoring="neg_mean_absolute_error", random_state=cb.SEED, n_jobs=1)
search.fit(X_train, y_train, groups=g_train)
results = (pd.DataFrame(search.cv_results_["params"])
             .assign(cv_mae=-search.cv_results_["mean_test_score"], std_over_folds=search.cv_results_["std_test_score"],
                     fit_seconds=search.cv_results_["mean_fit_time"])
             .sort_values("cv_mae"))
display(results.head(6).round(3))
print("Slechtste combinaties:")
results.tail(3).round(3)

,max_depth,max_features,max_samples,min_samples_leaf,cv_mae,std_over_folds,fit_seconds
9,18.0,0.902,0.776,9,30.919,2.155,2.369
0,18.0,0.858,0.428,8,31.565,2.027,1.430
2,18.0,0.621,0.534,8,32.390,2.021,1.287
1,NaN,0.718,0.612,23,34.284,2.029,1.381
5,NaN,0.732,0.728,29,34.783,2.042,1.510
7,25.0,0.505,0.556,15,35.272,1.975,0.998


Slechtste combinaties:


,max_depth,max_features,max_samples,min_samples_leaf,cv_mae,std_over_folds,fit_seconds
6,25.0,0.316,0.667,10,41.843,2.283,0.747
10,18.0,0.309,0.960,14,42.074,2.198,0.855
8,12.0,0.440,0.660,25,42.833,2.274,0.755


**Interpretatie ronde 1:** de tuning maakte het model **slechter**. De beste combinatie haalt CV-MAE 30,9, terwijl de standaardinstellingen 28,1 haalden (met 5 folds in plaats van 3, maar het verschil is groter dan de spreiding over de folds, ± 2). Het patroon in de tabel is duidelijk:
- de beste combinaties gebruiken **bijna alle kenmerken per split** (`max_features` 0,86–0,90) en **kleine bladeren** (8–9 uren);
- de slechtste gebruiken maar 30% van de kenmerken per split (MAE 42).

Dat verklaart het: de standaard random forest van scikit-learn voor regressie gebruikt **alle** kenmerken (`max_features = 1.0`) en bladeren van **1** uur. Onze zoekruimte sloot precies dat gebied uit. Met maar 9 kenmerken, waarvan 3 (uur, zone, temperatuur) het meeste werk doen, is een split zonder `hour` of `zone` vaak een slechte split. En grote bladeren middelen het zone × uur-detail weg.

### Ronde 2: zoeken rond de standaardinstellingen
We zoeken opnieuw, nu in de buurt van wat werkt: kleine bladeren (1–7), veel kenmerken per split (60–100%), weinig of geen dieptebeperking. Deze keer met dezelfde 5 folds als alle andere modellen (een fit duurt maar 1–2 seconden).

In [4]:
search2 = RandomizedSearchCV(
    RandomForestRegressor(n_estimators=150, n_jobs=-1, random_state=cb.SEED),
    {"min_samples_leaf": randint(1, 8), "max_features": uniform(0.6, 0.4),
     "max_depth": [None, 25, 35], "max_samples": uniform(0.5, 0.5)},
    n_iter=15, cv=GroupKFold(5), scoring="neg_mean_absolute_error", random_state=cb.SEED, n_jobs=1)
search2.fit(X_train, y_train, groups=g_train)
results2 = (pd.DataFrame(search2.cv_results_["params"])
              .assign(cv_mae=-search2.cv_results_["mean_test_score"], std_over_folds=search2.cv_results_["std_test_score"],
                      fit_seconds=search2.cv_results_["mean_fit_time"])
              .sort_values("cv_mae"))
results2.head(6).round(3)

,max_depth,max_features,max_samples,min_samples_leaf,cv_mae,std_over_folds,fit_seconds
5,25.0,0.810,0.716,1,27.477,2.172,4.177
9,35.0,0.872,0.725,2,27.617,2.202,3.479
10,NaN,0.986,0.904,1,27.688,2.161,5.414
7,35.0,0.782,0.893,3,27.761,2.136,3.620
12,35.0,0.844,0.917,3,27.762,2.180,3.620
4,25.0,0.600,0.996,1,27.771,2.079,3.851


**Interpretatie ronde 2:** nu liggen alle beste combinaties rond CV-MAE 27,5 tot 27,8, beter dan de standaard. De winnaars hebben **bladeren van 1 tot 3 uren**, **60–100% van de kenmerken** per split en een **bootstrap-steekproef van ± 70–90%** van de rijen. Dat laatste is de enige echte verbetering ten opzichte van de standaard: iets kleinere steekproeven per boom maken de bomen onderling diverser, en het gemiddelde van diversere bomen is beter. De verschillen tussen de top-6 (0,3) zijn wel klein ten opzichte van de spreiding over de folds (± 2,2).

## 3. Standaard of getuned? Cross-validatie en eindtest
We vergelijken de standaardinstellingen en de beste combinatie uit ronde 2 op exact dezelfde 5 folds, elk met 300 bomen, en houden het model met de laagste CV-MAE. (Zoals PyCaret's `choose_better=True`: tunen mag het model nooit slechter maken.)

In [5]:
candidates = {"standaard": {}, "getuned (ronde 2)": search2.best_params_}
scores = {name: cb.cv_scores(RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=cb.SEED, **params),
                             X_train, y_train, g_train)
          for name, params in candidates.items()}
display(pd.DataFrame({name: {"cv_mae": sc["cv_mae_mean"], **{f"fold {i}": m for i, m in enumerate(sc["cv_mae"])}}
                      for name, sc in scores.items()}).T)

chosen = min(scores, key=lambda name: scores[name]["cv_mae_mean"])
params = {k: (float(v) if isinstance(v, float) else v) for k, v in candidates[chosen].items()}
best = RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=cb.SEED, **params).fit(X_train, y_train)
pred = best.predict(X_test)
test_metrics = cb.evaluate(y_test, pred)
metrics = {"notebook": "05d_random_forest", "model": f"Random forest ({chosen})", "params": params,
           "cv_mae_default": cv_default["cv_mae_mean"], "cv_mae_round1_best": round(float(results["cv_mae"].min()), 4),
           **scores[chosen], **{f"test_{k}": v for k, v in test_metrics.items()}}
cb.save_results("random_forest", metrics, table.loc[y_test.index, ["zone", "time", "trips"]].assign(pred=pred))
print("Gekozen:", chosen, params)
pd.Series(test_metrics)

,cv_mae,fold 0,fold 1,fold 2,fold 3,fold 4
standaard,28.0527,27.9175,30.7164,24.5507,30.1065,26.9723
getuned (ronde 2),27.3988,27.1901,29.8627,23.9975,29.6030,26.3408


Gekozen: getuned (ronde 2) {'max_depth': 25, 'max_features': 0.8099025726528951, 'max_samples': 0.7159725093210578, 'min_samples_leaf': 1}


mae                 30.3160
rmse                58.9400
r2                   0.8967
poisson_deviance    13.9710
dtype: float64

## 4. Welke kenmerken gebruikt de random forest?
De ingebouwde *impurity importance* telt hoeveel elke variabele de fout verlaagt over alle splits. (Die bevoordeelt variabelen met veel waarden; in `06_model_comparison` gebruiken we daarom ook *permutation importance* op de testset.)

In [6]:
pd.Series(best.feature_importances_, index=cb.FEATURES).sort_values(ascending=False).round(3)

hour                0.396
zone                0.337
temperature_c       0.144
month               0.033
weekday             0.031
precipitation_mm    0.021
is_weekend          0.017
wind_kmh            0.015
is_holiday          0.005
dtype: float64

**Interpretatie:** `hour` (40%) en `zone` (34%) doen samen driekwart van het werk, daarna `temperature_c` (14%). Neerslag lijkt met 2% onbelangrijk, maar dat is misleidend: het regent maar in een klein deel van de uren (02 §7: 565 natte uren van 8.784). De *impurity importance* meet hoeveel een kenmerk gemiddeld over alle splits helpt, en een kenmerk dat zelden iets doet maar dan veel (regen) scoort daardoor laag. In `06` meten we met *permutation importance* hoeveel de fout echt stijgt zonder elk kenmerk.

## Samenvatting
| Variant | CV-MAE | Test-MAE | Test-R² |
|---|---|---|---|
| standaard (100 bomen) | 28,1 | | |
| ronde 1: beste uit brede zoekruimte (3 folds) | 30,9 | | |
| **ronde 2: getuned** (`min_samples_leaf=1`, `max_features≈0,81`, `max_samples≈0,72`, `max_depth=25`, 300 bomen) | **27,4** | **30,3** | **0,897** |

- De random forest doet het duidelijk beter dan de baseline (59) en het GLM (33,2 op de test), maar haalt de getunede gradient boosting (05e) niet.
- **Les:** een zoekruimte moet de standaardinstellingen bevatten en rond het werkende gebied liggen. De eerste ronde maakte het model slechter. We tonen dat bewust: zonder vergelijking met de standaard hadden we een slechter model "getuned" genoemd.
- Praktisch nadeel voor de deployment: 300 bomen tot diepte 25 met bladeren van 1 uur hebben elk tienduizenden knopen; als bestand is dat vele malen groter dan het boosting-model (± 5 MB) en trager om te laden op de server.